In [10]:
import pandas as pd
import numpy as np
import re

In [14]:
# --- 1. THE BULLETPROOF ATTRIBUTE PARSER ---
def parse_yelp_attributes(attr_dict):
    """Uses Regex to safely extract vibes, bypassing Yelp's broken dictionary formatting."""
    if not isinstance(attr_dict, dict):
        return ""
    
    extracted_features = []
    
    # 1. Extract Noise Level
    if 'NoiseLevel' in attr_dict and attr_dict['NoiseLevel']:
        noise = attr_dict['NoiseLevel'].replace("u'", "").replace("'", "")
        if noise.lower() not in ['none', '']:
            extracted_features.append(f"Noise level is {noise}.")
            
    # 2. Extract Alcohol
    if 'Alcohol' in attr_dict and attr_dict['Alcohol']:
        alc = attr_dict['Alcohol'].replace("u'", "").replace("'", "")
        if alc in ['full_bar', 'beer_and_wine']:
            extracted_features.append(f"Serves {alc.replace('_', ' ')}.")
            
    # 3. Extract Ambience using pure Regex
    if 'Ambience' in attr_dict and attr_dict['Ambience']:
        amb_str = str(attr_dict['Ambience'])
        # This finds any word wrapped in quotes that is followed immediately by : True
        true_vibes = re.findall(r"['\"]([a-zA-Z]+)['\"]\s*:\s*True", amb_str)
        if true_vibes:
            extracted_features.append(f"Ambience is {', '.join(true_vibes)}.")
            
    return " ".join(extracted_features)

In [15]:
# --- 2. PROCESS BUSINESS DATA & TIGHTEN CITY SWAP ---
print("Loading and augmenting business data...")
biz_df = pd.read_json('yelp/yelp_academic_dataset_business.json', lines=True)

# Filter for Philly Restaurants/Bars
philly_biz = biz_df[biz_df['city'] == 'Philadelphia'].copy()
philly_biz = philly_biz.dropna(subset=['categories'])
philly_biz = philly_biz[philly_biz['categories'].str.contains('Restaurants|Food|Bars')]

# Apply the new Regex parser!
philly_biz['metadata_sentence'] = philly_biz['attributes'].apply(parse_yelp_attributes)

# THE TIGHT CITY SWAP: Manhattan 
np.random.seed(42)
philly_biz['lat'] = np.random.uniform(40.7000, 40.8750, len(philly_biz))
philly_biz['lon'] = np.random.uniform(-74.0150, -73.9200, len(philly_biz))

philly_biz = philly_biz[['business_id', 'name', 'lat', 'lon', 'metadata_sentence']]
valid_business_ids = set(philly_biz['business_id'])

print(f"Augmented {len(philly_biz)} venues and mapped them to strict NYC coordinates.")

Loading and augmenting business data...
Augmented 7353 venues and mapped them to strict NYC coordinates.


In [16]:
# --- 3. PROCESS 5GB REVIEW FILE ---
print("Processing 5GB review file in chunks... (This takes a few minutes)")
chunk_size = 100000
filtered_reviews_list = []
chunks_processed = 0

for chunk in pd.read_json('/Users/alinazim/Downloads/yelp_dataset/yelp_academic_dataset_review.json', lines=True, chunksize=chunk_size):
    matched_reviews = chunk[chunk['business_id'].isin(valid_business_ids)].copy()
    
    if not matched_reviews.empty:
        # Rename the columns right now so we don't lose the text later!
        matched_reviews = matched_reviews.rename(columns={'stars': 'review_stars', 'text': 'review_text'})
        clean_chunk = matched_reviews[['business_id', 'review_stars', 'review_text']]
        filtered_reviews_list.append(clean_chunk)
    
    chunks_processed += 1
    if chunks_processed % 10 == 0:
        print(f"Processed {chunks_processed * chunk_size:,} rows...")

all_filtered_reviews = pd.concat(filtered_reviews_list, ignore_index=True)
print(f"Extraction complete! Kept {len(all_filtered_reviews):,} reviews.")

Processing 5GB review file in chunks... (This takes a few minutes)
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Processed 6,000,000 rows...
Processed 7,000,000 rows...
Extraction complete! Kept 755,335 reviews.


In [ ]:
# --- 4. MERGE AND FUSE (SAFELY) ---
print("Fusing metadata with review text...")
final_nlp_dataset = pd.merge(all_filtered_reviews, philly_biz, on='business_id', how='inner')

# FIX: Force Pandas to treat these strictly as strings, replacing missing values with blank spaces
final_nlp_dataset['metadata_sentence'] = final_nlp_dataset['metadata_sentence'].fillna("").astype(str)
final_nlp_dataset['review_text'] = final_nlp_dataset['review_text'].fillna("").astype(str)

# Glue them together with a clear divider so the NLP model understands the transition
final_nlp_dataset['augmented_text'] = final_nlp_dataset['metadata_sentence'] + " | Review: " + final_nlp_dataset['review_text']

# Let's keep the raw review_text in the DataFrame this time so you can inspect it!
final_nlp_dataset = final_nlp_dataset[['business_id', 'name', 'lat', 'lon', 'review_stars', 'augmented_text']]

# Save the final ML-ready file
final_nlp_dataset.to_csv('sweetspot_nlp_training_data.csv', index=False)
print("\nSuccess! Saved 'sweetspot_nlp_training_data.csv'.")


Fusing metadata with review text...

Success! Saved 'sweetspot_nlp_training_data.csv'.


,business_id,name,lat,lon,review_stars,augmented_text
0,kxX2SOes4o-D3ZQBkiMRfA,Zaika,40.743793,-73.931131,5,Noise level is average. Ambience is casual. | ...
1,04UD14gamNjLY0IDYVhHJg,Dmitri's,40.774744,-73.932203,1,Noise level is average. Ambience is casual. | ...
2,RZtGWDLCAtuipwaZ-UfjmQ,LaScala's,40.833264,-73.925249,4,Noise level is average. Serves full bar. Ambie...


In [23]:
print(pd.read_csv('sweetspot_nlp_training_data.csv').head(10))

              business_id                        name        lat        lon  \
0  kxX2SOes4o-D3ZQBkiMRfA                       Zaika  40.743793 -73.931131   
1  04UD14gamNjLY0IDYVhHJg                    Dmitri's  40.774744 -73.932203   
2  RZtGWDLCAtuipwaZ-UfjmQ                   LaScala's  40.833264 -73.925249   
3  YtSqYv1Q_pOltsVPSx54SA           Rittenhouse Grill  40.745193 -73.958311   
4  eFvzHawVJofxSnD7TgbZtg             Good Karma Cafe  40.774932 -73.988298   
5  kq5Ghhh14r-eCxlVmlyd8w           The Coventry Deli  40.754549 -73.991259   
6  oBhJuukGRqPVvYBfTkhuZA                 Square 1682  40.719259 -73.951596   
7  EtKSTHV5Qx_Q7Aur9o4kQQ             Village Whiskey  40.791054 -73.993991   
8  VJEzpfLs_Jnzgqh5A_FVTg  Jasmine Rice - Rittenhouse  40.705338 -73.990044   
9  NQSnr4RPUScss607oxOaqw            Chase's Hop Shop  40.715486 -73.958274   

   review_stars                                     augmented_text  
0             5  Noise level is average. Ambience is casual. 

In [29]:
df = pd.read_csv('yelp/sweetspot_nlp_training_data.csv')

# --- TEST 1: Macro Statistics & Missing Values ---
print(f"Total Reviews Extracted: {len(df):,}")
print(f"Total Unique Venues: {df['business_id'].nunique():,}")

Total Reviews Extracted: 755,335
Total Unique Venues: 7,353


In [27]:
print("\n--- Review Star Distribution ---")
# Calculate the percentage of reviews for each star rating
star_dist = df['review_stars'].value_counts(normalize=True).sort_index() * 100
for star, pct in star_dist.items():
    print(f"{int(star)} Star: {pct:.1f}%")


--- Review Star Distribution ---
1 Star: 9.9%
2 Star: 8.3%
3 Star: 13.2%
4 Star: 28.1%
5 Star: 40.6%


In [30]:
print("\n--- Augmented Text Sample ---")
sample_text = df['augmented_text'].iloc[0] # Grab the very first row
print(sample_text[:300] + "..." if len(sample_text) > 300 else sample_text)


--- Augmented Text Sample ---
Noise level is average. Ambience is casual. | Review: Wow!  Yummy, different,  delicious.   Our favorite is the lamb curry and korma.  With 10 different kinds of naan!!!  Don't let the outside deter you (because we almost changed our minds)...go in and try something new!   You'll be glad you did!
